# 61 — Passive scan: rilevamento per canale

**Perché.** Nel notebook 60 il passive scan sul canale 5745 MHz sembrava il regime difficile (AUROC one-class 0,62). In realtà ogni file passive è una **scansione sequenziale** dei canali: 5180 → 5825 MHz a 5 GHz, 2412 → 2462 MHz a 2,4 GHz, circa 4.700 righe. Le prime 128 righe usate dal protocollo contengono **solo il primo canale**, quindi un jammer a 5745 MHz non vi compare mai.

**Cosa fa questo notebook.**
- Per ogni file, finestre di W = 8, 32, 128 righe consecutive **per canale**.
- Per ogni canale monitorato, rilevatori one-class addestrati sui benigni a quel canale (modelli e soglia conforme come nel protocollo del relatore) e Random Forest supervisionato come riferimento.
- **Matrice canale monitorato × canale del jammer:**
  - la diagonale è il rilevamento vero;
  - i valori alti lontano dalla diagonale indicano **scorciatoie**: differenze di sessione o acquisizione tra file benigni e malevoli, non il jamming.
- **Calibrazione ampia:** tutti i file di fondo in camera (20.432), divisi per file 50/25/25, per stimare soglie anche a α = 0,1%.
- **Controllo fuori distribuzione:** soglie calibrate in camera, testate su benigni di ambiente reale.

In [ ]:
import os, sys, time, json, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
from multiprocessing import Pool
from qdetect.files import build_manifest
from qdetect import passive as P
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 300)
DATA_ROOT = pathlib.Path(os.environ.get('QS_DATA_ROOT', '/home/habes/NotebookProjects/bboi/monaco/rf_jamming_dataset'))
N_JOBS = int(os.environ.get('N_JOBS', '16'))
BG_PER_GROUP = int(os.environ.get('BG_PER_GROUP', '1500'))   # benigni ambiente reale per (banda, luogo)
OUT = REPO / 'results' / 'passive_canali'; OUT.mkdir(parents=True, exist_ok=True)
man = build_manifest(DATA_ROOT)
man = man[man.scan_mode == 'passive'].reset_index(drop=True)
bg = man[(man.label == 'benign') & (man.benign_subtype == 'background')]
bg = bg.groupby(['band', 'location'], group_keys=False).apply(lambda g: g.sample(min(len(g), BG_PER_GROUP), random_state=0))
sel = pd.concat([man[man.label == 'malicious'], man[(man.label == 'benign') & (man.benign_subtype == 'floor')], bg]).reset_index(drop=True)
sel['split'] = P.file_split(sel.file_path.tolist())
print(sel.groupby(['band', 'label', 'benign_subtype', 'split']).size().unstack())

## 1. Finestre per canale (lettura parallela dei file)

In [ ]:
def work(rel):
    d = pd.read_csv(DATA_ROOT / rel)
    return [dict(r, file_path=rel) for r in P.channel_windows(d)]
t0 = time.time()
with Pool(N_JOBS) as pool:
    recs = [r for part in pool.imap_unordered(work, sel.file_path.tolist(), chunksize=64) for r in part]
F = pd.DataFrame(recs).merge(sel, on='file_path')
num = F.select_dtypes('float64').columns; F[num] = F[num].astype('float32')
print(f'{len(sel)} file -> {len(F)} finestre in {time.time()-t0:.0f} s')
F.to_pickle(OUT / 'finestre_canale.pkl.gz')

## 2. Matrice canale monitorato × canale del jammer (W = 128, medie, 7 feature)

In [ ]:
t0 = time.time(); parts = []
for band in ['5ghz', '2.4ghz']:
    for fields in [P.ALL7, P.CORE]:
        parts.append(P.channel_matrix(F, band, 128, fields, 'mean'))
M = pd.concat(parts); M.to_csv(OUT / 'matrice_canali_W128.csv', index=False)
print(f'{time.time()-t0:.0f} s, {len(M)} righe')
best = M[M.fields == 'all7'].groupby(['band', 'training', 'monitored_mhz', 'jammer_mhz']).auroc.max().reset_index()
for band in ['5ghz', '2.4ghz']:
    for tr in ['benign_only', 'supervised']:
        print(f'\n=== {band}, {tr}: AUROC max (righe = canale monitorato, colonne = jammer) ===')
        display(best[(best.band == band) & (best.training == tr)].pivot(index='monitored_mhz', columns='jammer_mhz', values='auroc').round(3))

## 3. Rilevamento vero: canale monitorato = canale del jammer

Per ogni canale del jammer e ogni potenza: miglior modello one-class e supervisionato, TPR ai livelli α = 1% e 0,1%, FPR osservato e limite superiore al 95%.

In [ ]:
D = M[M.monitored_mhz == M.jammer_mhz]
cols = ['band', 'jammer_mhz', 'power_dbm', 'fields', 'training', 'model', 'auroc', 'tpr@0.01', 'fpr@0.01', 'fpr_up95@0.01', 'tpr@0.001', 'fpr@0.001', 'fpr_up95@0.001', 'n_cal', 'n_neg', 'n_pos']
top = D.sort_values('tpr@0.01').groupby(['band', 'jammer_mhz', 'power_dbm', 'fields', 'training']).tail(1)
display(top[cols].sort_values(['band', 'jammer_mhz', 'power_dbm', 'fields', 'training']).round(4))
top[cols].to_csv(OUT / 'diagonale_W128.csv', index=False)

## 4. Scorciatoie: canali lontani dal jammer

Canale monitorato a più di 40 MHz dal jammer (2,4 GHz: più di 25 MHz, perché i canali sono larghi 20 MHz e si sovrappongono). Qui un'AUROC lontana da 0,5 non può venire dal jamming.

In [ ]:
M['dist'] = (M.monitored_mhz - M.jammer_mhz).abs()
far = M[((M.band == '5ghz') & (M.dist > 40)) | ((M.band == '2.4ghz') & (M.dist > 25))]
sc = far.groupby(['band', 'fields', 'training', 'jammer_mhz']).auroc.agg(['median', 'max']).round(3)
display(sc); sc.to_csv(OUT / 'scorciatoie_W128.csv')

## 5. Budget di report (W = 8, 32, 128) e rappresentazione, sulla diagonale

In [ ]:
parts = []
for band in ['5ghz', '2.4ghz']:
    for W in [8, 32, 128]:
        for kind in ['mean', 'second']:
            if W == 128 and kind == 'mean':
                continue
            m = P.channel_matrix(F, band, W, P.ALL7, kind)
            parts.append(m[m.monitored_mhz == m.jammer_mhz])
Bd = pd.concat(parts + [M[(M.monitored_mhz == M.jammer_mhz) & (M.fields == 'all7')]]); Bd.to_csv(OUT / 'budget_diagonale.csv', index=False)
bt = Bd[Bd.training == 'benign_only'].groupby(['band', 'jammer_mhz', 'power_dbm', 'W', 'kind'])[['auroc', 'tpr@0.01']].max().unstack(['W', 'kind']).round(3)
display(bt)

## 6. Fuori distribuzione: soglia calibrata in camera, benigni di test da ambiente reale

In [ ]:
parts = [P.channel_matrix(F, band, 128, P.ALL7, 'mean', test_benign_subtype='background') for band in ['5ghz', '2.4ghz']]
O = pd.concat(parts); O = O[O.monitored_mhz == O.jammer_mhz]; O.to_csv(OUT / 'ood_background_W128.csv', index=False)
display(O.sort_values('tpr@0.01').groupby(['band', 'jammer_mhz', 'power_dbm', 'training']).tail(1)[['band', 'jammer_mhz', 'power_dbm', 'training', 'model', 'auroc', 'tpr@0.01', 'fpr@0.01']].round(4))